In [26]:
import json
from pathlib import Path
from datetime import datetime, timezone
import yfinance as yf
import os
import requests
import pandas as pd

In [27]:


PROJECT_ROOT = Path.cwd().parent
COINS_PATH = PROJECT_ROOT / "data" / "raw" / "metadata" / "coins.json"

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "Sample"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "Sample"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [28]:


cfg = json.loads(COINS_PATH.read_text())

def fetch_yahoo_ohlcv(symbol: str, ticker: str, start_year: int):
    start = f"{start_year}-01-01"
    df = yf.download(ticker, start=start, interval="1d", auto_adjust=False, progress=False)

    # Flatten MultiIndex columns if present
    if hasattr(df.columns, "levels"):
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]

    df = df.reset_index()
    df.rename(columns={
        "Date": "date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume",
    }, inplace=True)

    df["symbol"] = symbol.upper()
    df["date"] = df["date"].dt.date.astype(str)

    out_df = df[["symbol", "date", "open", "high", "low", "close", "volume"]].copy()
    return out_df

ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

dfs = {}

for c in cfg["coins"]:
    sym = c["symbol"].upper()
    ticker = c["yahoo_ticker"]
    start_year = int(c["start_year"])

    out_df = fetch_yahoo_ohlcv(sym, ticker, start_year)
    dfs[sym] = out_df



    print(sym, "rows:", len(out_df), "range:", out_df["date"].min(), "->", out_df["date"].max())


btc_df = dfs.get("BTC")
eth_df = dfs.get("ETH")

print("\nBTC tail:")
display(dfs.get("BTC").tail(3))

print("\nETH tail:")
display(dfs.get("ETH").tail(3))


BTC rows: 4163 range: 2014-09-17 -> 2026-02-08
ETH rows: 3014 range: 2017-11-09 -> 2026-02-08

BTC tail:


,symbol,date,open,high,low,close,volume
4160,BTC,2026-02-06,62704.453125,71681.304688,60074.203125,70555.390625,114674259489
4161,BTC,2026-02-07,70553.796875,71611.148438,67364.445312,69281.968750,62347107663
4162,BTC,2026-02-08,69302.406250,69566.585938,68945.664062,69295.992188,56693329920



ETH tail:


,symbol,date,open,high,low,close,volume
3011,ETH,2026-02-06,1821.129395,2090.328125,1748.627686,2063.386963,64998454203
3012,ETH,2026-02-07,2063.333740,2115.824951,1995.918579,2090.549316,41145430304
3013,ETH,2026-02-08,2089.215576,2101.961670,2072.165283,2077.603760,36837888000


In [29]:



BASE = "https://api.coingecko.com/api/v3"



cfg = json.loads(COINS_PATH.read_text())

def get_json(url, params=None):
    headers = {}
    key = os.getenv("COINGECKO_API_KEY") # in Demo replace this line with my actual API key , if needed
    if key:
        headers["x-cg-demo-api-key"] = key
    r = requests.get(url, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_mcap_1y(coin_id: str, vs: str = "usd") -> pd.DataFrame:
    data = get_json(
        f"{BASE}/coins/{coin_id}/market_chart",
        params={"vs_currency": vs, "days": "365", "interval": "daily"},
    )
    mcap = pd.DataFrame(data["market_caps"], columns=["ts_ms", "market_cap"])
    mcap["date"] = pd.to_datetime(mcap["ts_ms"], unit="ms", utc=True).dt.date.astype(str)
    mcap = mcap.sort_values("ts_ms").groupby("date", as_index=False).last()
    return mcap[["date", "market_cap"]]

vs = cfg.get("vs_currency", "usd")

final_dfs = {}

for c in cfg["coins"]:
    sym = c["symbol"].upper()
    coin_id = c["coingecko_id"]

    print("\nProcessing", sym)

    ydf = dfs[sym].copy()

    
    try:
        mcap_df = fetch_mcap_1y(coin_id, vs)
        merged = ydf.merge(mcap_df, on="date", how="left")
        coverage = round(100 * merged["market_cap"].notna().mean(), 2)
        print("  market_cap coverage:", coverage, "% (expected: about 365 days only)")
    except Exception as e:
        
        merged = ydf.copy()
        merged["market_cap"] = pd.NA
        print("  market_cap fetch failed, leaving empty. Error:", type(e).__name__)

    merged = merged[["symbol", "date", "open", "high", "low", "close", "volume", "market_cap"]]
    final_dfs[sym] = merged


print("\nBTC tail:")
display(final_dfs["BTC"].tail(3))

print("\nETH tail:")
display(final_dfs["ETH"].tail(3))



Processing BTC
  market_cap coverage: 8.77 % (expected: about 365 days only)

Processing ETH
  market_cap coverage: 12.11 % (expected: about 365 days only)

BTC tail:


,symbol,date,open,high,low,close,volume,market_cap
4160,BTC,2026-02-06,62704.453125,71681.304688,60074.203125,70555.390625,114674259489,1.262033e+12
4161,BTC,2026-02-07,70553.796875,71611.148438,67364.445312,69281.968750,62347107663,1.410204e+12
4162,BTC,2026-02-08,69302.406250,69566.585938,68945.664062,69295.992188,56693329920,1.384945e+12



ETH tail:


,symbol,date,open,high,low,close,volume,market_cap
3011,ETH,2026-02-06,1821.129395,2090.328125,1748.627686,2063.386963,64998454203,2.221735e+11
3012,ETH,2026-02-07,2063.333740,2115.824951,1995.918579,2090.549316,41145430304,2.487488e+11
3013,ETH,2026-02-08,2089.215576,2101.961670,2072.165283,2077.603760,36837888000,2.509303e+11


In [30]:


for sym, df in final_dfs.items():
    df = df.sort_values("date").copy()

# TODO : We need to decide what we want to do with NaN values for market cap    
#    df["market_cap"] = df["market_cap"].bfill()

    out_path = PROCESSED_DIR / f"{sym}_daily.csv"
    df.to_csv(out_path, index=False)
    print(sym, "saved:", out_path.resolve())

# quick check
display(final_dfs["BTC"].sort_values("date").head(3))
display(final_dfs["BTC"].sort_values("date").tail(3))


BTC saved: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service\data\processed\Sample\BTC_daily.csv
ETH saved: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service\data\processed\Sample\ETH_daily.csv


,symbol,date,open,high,low,close,volume,market_cap
0,BTC,2014-09-17,465.864014,468.174011,452.421997,457.334015,21056800,NaN
1,BTC,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200,NaN
2,BTC,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700,NaN


,symbol,date,open,high,low,close,volume,market_cap
4160,BTC,2026-02-06,62704.453125,71681.304688,60074.203125,70555.390625,114674259489,1.262033e+12
4161,BTC,2026-02-07,70553.796875,71611.148438,67364.445312,69281.968750,62347107663,1.410204e+12
4162,BTC,2026-02-08,69302.406250,69566.585938,68945.664062,69295.992188,56693329920,1.384945e+12
